In [1]:
import sys
from pathlib import Path

# Get project root - the project root should have both 'vault' and 'ensemble' directories
# Walk up from current directory until we find a directory with both
current_dir = Path.cwd()
project_root = current_dir

# Walk up the directory tree to find project root
while project_root != project_root.parent:
    # Check if this directory has both vault and ensemble (indicating project root)
    has_vault = (project_root / 'vault').exists()
    has_ensemble = (project_root / 'ensemble').exists()
    
    if has_vault and has_ensemble:
        # Found the project root
        break
    
    project_root = project_root.parent
else:
    # If we reached the filesystem root without finding it, use parent of current_dir
    # (assuming we're in research/ subdirectory)
    if (current_dir.parent / 'vault').exists() and (current_dir.parent / 'ensemble').exists():
        project_root = current_dir.parent
    else:
        # Last resort: use current directory
        project_root = current_dir

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Has vault: {(project_root / 'vault').exists()}")
print(f"Has ensemble: {(project_root / 'ensemble').exists()}")
print(f"Python path includes: {str(project_root) in sys.path}")

Current directory: /home/raman/repos/Trading-Algo/research
Project root: /home/raman/repos/Trading-Algo
Has vault: True
Has ensemble: True
Python path includes: True


In [2]:
"""
Portfolio Testing with PortfolioTester

This notebook demonstrates how to use the PortfolioTester framework to:
1. Load data and create portfolios
2. Fit portfolios using candles
3. Generate predictions at different granularity levels
4. Calculate returns and generate tearsheets
5. Analyze performance at portfolio, ensemble, and base model levels
"""

from datetime import datetime
import pandas as pd
import numpy as np

from utils.enums import TimeFrame, Ticker
from utils import helpers
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.portfolio_tester import PortfolioTester

# Portfolio Testing Framework Demo

## Overview

The `PortfolioTester` class provides a comprehensive framework for testing portfolios:
- **Automatic returns calculation**: Calculates log returns from candles automatically
- **Granular predictions**: Get predictions at portfolio, ensemble, and base model levels
- **Integrated tearsheets**: Generate QuantStats tearsheets at all granularity levels
- **Baseline comparison**: Compare against equal-weight or buy-and-hold baselines

In [3]:
# Load data
print("Loading data...")
train_candles = helpers.load_data_multi_ticker(
    tickers=[Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY],
    timeframe=TimeFrame.D,
    start=datetime(2010, 1, 1),
    end=datetime(2020, 1, 1),
    use_millisecond_offset=True
)

test_candles = helpers.load_data_multi_ticker(
    tickers=[Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY],
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 1, 1),
    use_millisecond_offset=True
)

print(f"Training data: {len(train_candles)} candles")
print(f"Test data: {len(test_candles)} candles")
print(f"Tickers: {sorted(train_candles['ticker'].unique())}")

Loading data...
Training data: 8220 candles
Test data: 4032 candles
Tickers: [<Ticker.ES: 'US500'>, <Ticker.NQ: 'US100'>, <Ticker.YM: 'US30'>, <Ticker.RTY: 'RUSSELL_2000'>]


## Step 1: Create Portfolio

For this demo, we'll use a simple buy-and-hold ensemble. In practice, you would load your ensemble from the vault or create it from feature configs.

In [4]:
# Load ensemble from vault using vault_manager
from ensemble.vault_manager import load_ensemble_from_vault

# Load ensemble from vault
ensemble_dir = 'vault/D/buy_hold_long'
print(f"Loading ensemble from {ensemble_dir}...")
ensemble = load_ensemble_from_vault(ensemble_dir, refit=False, target_volatility=0.10)
print(f"Ensemble loaded: {ensemble}")
print(f"Ensemble fitted: {ensemble.is_fitted_}")
print(f"Base models: {list(ensemble.base_models.keys())}")

# Create portfolio
portfolio = Portfolio(
    ensembles=[ensemble],
    trading_timeframe=TimeFrame.D,
    target_volatility=0.10,
    max_position_pct=2.0
)
print(f"\nPortfolio created with {len(portfolio.ensembles)} ensemble(s)")

Loading ensemble from vault/D/buy_hold_long...
Ensemble loaded: DiversifiedEnsemble (fitted)
  Target Volatility: 0.1
  Features: 1
  Tickers: 4
  Top 3 weights: {'buy_hold_signal_D::quantile_binning_2': 1.0}
Ensemble fitted: True
Base models: ['buy_hold_signal_D::quantile_binning_2']

Portfolio created with 1 ensemble(s)


## Step 2: Initialize PortfolioTester

The PortfolioTester wraps your portfolio and provides a convenient API for testing.

In [5]:
# Initialize PortfolioTester
tester = PortfolioTester(
    portfolio=portfolio,
    baseline_mode='equal_weight'  # or 'buy_hold'
)

# Fit portfolio (automatically calculates log returns)
print("\nFitting portfolio...")
tester.fit(train_candles)

print(f"Portfolio fitted: {tester.portfolio.is_fitted_}")
print(f"IDM: {tester.portfolio.idm_}")
print(f"Ensemble fitted: {ensemble.is_fitted_}")


Fitting portfolio...
Portfolio fitted: True
IDM: None
Ensemble fitted: True


## Step 3: Generate Predictions

You can get predictions at different granularity levels:
- **Portfolio level**: Aggregated positions (default)
- **Ensemble level**: Individual ensemble predictions
- **Base model level**: Individual base model predictions

In [6]:
# Basic prediction (portfolio level only)
print("\nGenerating portfolio-level predictions...")
positions = tester.predict(test_candles)
print(f"Generated {len(positions)} position predictions")
print(f"Position fraction range: [{positions['position_fraction'].min():.4f}, {positions['position_fraction'].max():.4f}]")

# With ensemble-level and base model-level granularity
print("\nGenerating granular predictions (ensemble + base model level)...")
results = tester.predict(
    test_candles,
    return_ensemble_predictions=True,
    return_base_model_predictions=True
)

portfolio_positions = results['portfolio']
ensemble_predictions = results['ensembles']
base_model_predictions = results['base_models']

print(f"\nGranular predictions:")
print(f"  Portfolio positions: {len(portfolio_positions)}")
print(f"  Ensemble predictions: {len(ensemble_predictions)}")
print(f"  Base model predictions: {len(base_model_predictions)}")


Generating portfolio-level predictions...
Generated 4032 position predictions
Position fraction range: [0.1393, 0.4898]

Generating granular predictions (ensemble + base model level)...

Granular predictions:
  Portfolio positions: 4032
  Ensemble predictions: 1
  Base model predictions: 1


In [7]:
# Debug: Check what the ensemble is actually returning
print("Debugging ensemble predictions...")
ensemble_pred = ensemble.predict_from_candles(test_candles)
print(f"Ensemble prediction type: {type(ensemble_pred)}")
print(f"Ensemble prediction shape: {ensemble_pred.shape if isinstance(ensemble_pred, pd.DataFrame) else 'N/A'}")
if isinstance(ensemble_pred, pd.DataFrame):
    print(f"Ensemble prediction columns: {ensemble_pred.columns.tolist()}")
    print(f"Ensemble forecast_score range: [{ensemble_pred['forecast_score'].min():.4f}, {ensemble_pred['forecast_score'].max():.4f}]")
    print(f"Ensemble forecast_score non-zero count: {(ensemble_pred['forecast_score'] != 0).sum()}")
    print("\nFirst few ensemble predictions:")
    print(ensemble_pred.head(10))
    
    # Check base model predictions
    print("\n\nChecking base model predictions...")
    base_model_name = list(ensemble.base_models.keys())[0]
    print(f"Base model: {base_model_name}")
    
    # Test base model prediction for one ticker
    test_ticker = test_candles['ticker'].iloc[0]
    ticker_candles = test_candles[test_candles['ticker'] == test_ticker].copy()
    base_model = ensemble.base_models[base_model_name]
    base_pred = base_model.predict(ticker_candles)
    print(f"Base model prediction type: {type(base_pred)}")
    print(f"Base model prediction shape: {base_pred.shape if hasattr(base_pred, 'shape') else len(base_pred)}")
    print(f"Base model prediction range: [{base_pred.min():.4f}, {base_pred.max():.4f}]")
    print(f"Base model non-zero count: {(base_pred != 0).sum()}")
    print(f"\nFirst 10 base model predictions:")
    print(base_pred.head(10))

Debugging ensemble predictions...
Ensemble prediction type: <class 'pandas.core.frame.DataFrame'>
Ensemble prediction shape: (4032, 3)
Ensemble prediction columns: ['ticker', 'datetime', 'forecast_score']
Ensemble forecast_score range: [0.5572, 1.9591]
Ensemble forecast_score non-zero count: 4032

First few ensemble predictions:
      ticker   datetime  forecast_score
0  Ticker.ES 2020-01-02        1.476734
1  Ticker.ES 2020-01-03        1.476734
2  Ticker.ES 2020-01-06        1.476734
3  Ticker.ES 2020-01-07        1.476734
4  Ticker.ES 2020-01-08        1.476734
5  Ticker.ES 2020-01-09        1.476734
6  Ticker.ES 2020-01-10        1.476734
7  Ticker.ES 2020-01-13        1.476734
8  Ticker.ES 2020-01-14        1.476734
9  Ticker.ES 2020-01-15        1.476734


Checking base model predictions...
Base model: buy_hold_signal_D::quantile_binning_2
Base model prediction type: <class 'pandas.core.series.Series'>
Base model prediction shape: (1008,)
Base model prediction range: [1.0000, 1.0

In [8]:
portfolio_positions

,ticker,datetime,forecast_score,position_fraction
0,Ticker.ES,2020-01-02 00:00:00.000,1.476734,0.369184
1,Ticker.NQ,2020-01-02 00:00:00.001,1.132120,0.283030
2,Ticker.YM,2020-01-02 00:00:00.002,1.959070,0.489767
3,Ticker.RTY,2020-01-02 00:00:00.003,0.557240,0.139310
4,Ticker.ES,2020-01-03 00:00:00.000,1.476734,0.369184
...,...,...,...,...
4027,Ticker.RTY,2023-12-28 00:00:00.003,0.557240,0.139310
4028,Ticker.ES,2023-12-29 00:00:00.000,1.476734,0.369184
4029,Ticker.NQ,2023-12-29 00:00:00.001,1.132120,0.283030
4030,Ticker.YM,2023-12-29 00:00:00.002,1.959070,0.489767


## Step 4: Calculate Returns

Calculate strategy returns and baseline returns for comparison.

In [9]:
# Calculate strategy returns
print("\nCalculating strategy returns...")
strategy_returns = tester.calculate_strategy_returns(test_candles)
print(f"Strategy returns: {len(strategy_returns)} days")
print(f"Total return: {strategy_returns.sum():.4f}")
if strategy_returns.std() > 0:
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {sharpe:.4f}")

# Calculate baseline returns
print("\nCalculating baseline returns...")
baseline_returns = tester.calculate_baseline_returns(test_candles)
print(f"Baseline returns: {len(baseline_returns)} days")
print(f"Total return: {baseline_returns.sum():.4f}")
if baseline_returns.std() > 0:
    baseline_sharpe = baseline_returns.mean() / baseline_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {baseline_sharpe:.4f}")

# Compare performance
print(f"\nPerformance Comparison:")
print(f"  Strategy total return: {strategy_returns.sum():.4f} ({strategy_returns.sum()*100:.2f}%)")
print(f"  Baseline total return: {baseline_returns.sum():.4f} ({baseline_returns.sum()*100:.2f}%)")
print(f"  Excess return: {(strategy_returns.sum() - baseline_returns.sum()):.4f} ({(strategy_returns.sum() - baseline_returns.sum())*100:.2f}%)")


Calculating strategy returns...
Strategy returns: 4028 days
Total return: 0.4919
Annualized Sharpe: 0.3790

Calculating baseline returns...
Baseline returns: 4028 days
Total return: 1.5205
Annualized Sharpe: 0.3724

Performance Comparison:
  Strategy total return: 0.4919 (49.19%)
  Baseline total return: 1.5205 (152.05%)
  Excess return: -1.0286 (-102.86%)


## Step 5: Generate Tearsheets

Generate QuantStats tearsheets for performance analysis.

In [12]:
# Generate portfolio-level tearsheet
print("\nGenerating portfolio tearsheet...")
tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio',
    output_dir='reports/portfolio',  # Save HTML report to this directory
    mode='html',  # 'html', 'full', 'basic', or 'metrics'
    candles_df=test_candles
)

# Generate ensemble-level tearsheets (if available)
if tester.ensemble_predictions:
    print("\nGenerating ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

# Generate base model-level tearsheets (if available)
if tester.base_model_predictions:
    print("\nGenerating base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )


Generating portfolio tearsheet...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/portfolio/Buy_Hold_Portfolio_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/ensembles/ensemble_0_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating base model tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/base_models/ensemble_0_buy_hold_signal_D_quantile_binning_2_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


## Complete Example Workflow

Here's a complete example showing the full workflow:

In [11]:
# Summary: Access all results
print("\n" + "="*70)
print("SUMMARY: All Results Available")
print("="*70)
print(f"Portfolio positions: {len(tester.positions_df) if tester.positions_df is not None else 0}")
print(f"Ensemble predictions: {len(tester.ensemble_predictions) if tester.ensemble_predictions else 0}")
print(f"Base model predictions: {len(tester.base_model_predictions) if tester.base_model_predictions else 0}")
print(f"Strategy returns: {len(tester.strategy_returns) if tester.strategy_returns is not None else 0}")
print(f"Baseline returns: {len(tester.baseline_returns) if tester.baseline_returns is not None else 0}")

# Example: Access specific results
if tester.positions_df is not None:
    print(f"\nPortfolio positions sample:")
    print(tester.positions_df.head())
    
if tester.strategy_returns is not None:
    print(f"\nStrategy returns statistics:")
    print(f"  Mean daily return: {tester.strategy_returns.mean():.6f}")
    print(f"  Std daily return: {tester.strategy_returns.std():.6f}")
    print(f"  Min return: {tester.strategy_returns.min():.6f}")
    print(f"  Max return: {tester.strategy_returns.max():.6f}")


SUMMARY: All Results Available
Portfolio positions: 4032
Ensemble predictions: 1
Base model predictions: 1
Strategy returns: 4028
Baseline returns: 4028

Portfolio positions sample:
       ticker                datetime  forecast_score  position_fraction
0   Ticker.ES 2020-01-02 00:00:00.000        1.476734           0.369184
1   Ticker.NQ 2020-01-02 00:00:00.001        1.132120           0.283030
2   Ticker.YM 2020-01-02 00:00:00.002        1.959070           0.489767
3  Ticker.RTY 2020-01-02 00:00:00.003        0.557240           0.139310
4   Ticker.ES 2020-01-03 00:00:00.000        1.476734           0.369184

Strategy returns statistics:
  Mean daily return: 0.000122
  Std daily return: 0.005115
  Min return: -0.058660
  Max return: 0.052930
